# Adapter un modèle au corpus kabyle avec Colab
Choisir **Exécution → Modifier le type d’exécution → GPU T4**. Aucun abonnement n’est souscrit par ce notebook.

Modèle candidat : `Qwen/Qwen2.5-0.5B-Instruct`. Sa qualité en kabyle doit être mesurée. On adapte ses traductions avec QLoRA ; ce corpus bilingue ne constitue pas un corpus de conversations natives.

Sur ton PC, après `git pull`, lance `bash scripts/export_chat_dataset.sh`. Importe ensuite le ZIP obtenu. Les traductions liées restent dans le même ensemble pour éviter les fuites entre entraînement et évaluation. Les données non validées restent signalées dans les métadonnées.


In [6]:
from google.colab import files
from pathlib import Path
import json, zipfile, subprocess, sys, hashlib
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Choisis uniquement l’archive ZIP de l’export.")
archive = Path(next(iter(uploaded)))
dataset = Path('/content/kabyle-dataset')
dataset.mkdir(exist_ok=True)
allowed = {'train.jsonl', 'validation.jsonl', 'test.jsonl', 'provenance.jsonl', 'manifest.json'}
with zipfile.ZipFile(archive) as z:
    if set(z.namelist()) != allowed or sum(x.file_size for x in z.infolist()) > 2_000_000_000:
        raise ValueError('Archive inattendue ou trop volumineuse.')
    z.extractall(dataset)
manifest = json.loads((dataset/'manifest.json').read_text())
print(manifest)


Saving 20261106T093950Z.zip to 20261106T093950Z.zip
{'version': 'kab-bilingual-sft-v1', 'base_model_candidate': 'Qwen/Qwen2.5-0.5B-Instruct', 'pairs': 233181, 'examples': {'train': 420028, 'validation': 22790, 'test': 23544}, 'pairs_by_split': {'train': 210014, 'validation': 11395, 'test': 11772}, 'excluded': {'duplicates': 1767, 'too_long': 68, 'empty': 1, 'needs_correction': 26}, 'sha256': {'train': '7c63a427ce7e84eb6f8546aa11bce6cb7bea6bf37ba90c2f340a96ee14fa206a', 'validation': '914490b2f9257a05142bf3aa70b4b2a7e7936207a921e3d9f81921bea41d3841', 'test': 'a847341df5f8ed016c0e5e3333ec4733b6a5350675088e2cd14fa34ecb8ad9ca', 'provenance': 'b1f1d5ab7f47bc8307e9dda9b414c894db8ac8cf7c65b457d863526c84b3a0fd'}, 'tasks': ['kabyle_to_french', 'french_to_kabyle'], 'dataset_kind': 'bilingual_instruction_templates_not_native_dialogues', 'split_policy': 'connected components of normalized Kabyle/French strings; bidirectional examples stay in the same split', 'training_status': 'not_started', 'quali

In [7]:
import re
revision = manifest.get('code_revision', '')
if not re.fullmatch(r'[0-9a-f]{40}', revision):
    raise ValueError('Réexporte le corpus avec le script du dépôt mis à jour : révision Git absente.')
repo = Path('/content/Kabyle')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/liliabkm/Kabyle.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'fetch', 'origin', revision], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '--detach', revision], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(repo/'training/requirements.txt')], check=True)
import torch
assert torch.cuda.is_available(), 'Active le GPU T4 dans les paramètres Colab.'
print(torch.cuda.get_device_name(0))


Tesla T4


## Sauvegarde et durée
Le mode `pilot` lance **200 étapes d’optimisation**, pas l’intégralité du corpus. Il sert à vérifier le pipeline. Passe à `full` pour une époque complète ; selon le corpus, cela peut dépasser une session Colab gratuite. Les checkpoints sont sauvegardés toutes les 100 étapes. Google Drive permet de reprendre après interruption, avec le même corpus et le même modèle.


In [9]:
import hashlib
import json
import sys
import subprocess
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
MODE = 'pilot'  # 'pilot' ou 'full'
BASE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
assert MODE in ('pilot', 'full')
run_id = hashlib.sha256(json.dumps(manifest['sha256'], sort_keys=True).encode()).hexdigest()[:12]
output = Path('/content/drive/MyDrive/KabyleChat') / (run_id + '-' + MODE)
command = [sys.executable, str(repo/'training/finetune.py'), '--dataset', str(dataset),
           '--output', str(output), '--base-model', BASE_MODEL,
           '--max-steps', '200' if MODE == 'pilot' else '-1']
if list(output.glob('checkpoint-*')):
    command.append('--resume')
process = subprocess.Popen(
    [command[0], "-u", *command[1:]],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
for line in process.stdout:
    print(line, end="", flush=True)
process.wait()
print("\nCode de sortie :", process.returncode)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
I0000 00:00:1793970759.876198    7765 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793970762.428960    7765 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.
Traceback (most recent call last):
  File "/content/Kabyle/training/finetune.py", line 105, in <module>
    if __name__=='__main__':main()
                            ~~~~^^
  File "/content/Kabyle/training/finetune.py", line 62, in main
    model=AutoModelForCausalLM.from_pretrained(a.base_model,revision=resolved,trust_remote_code=Fal

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "bitsandbytes"],
    check=True,
)
subprocess.run(
    ["git", "-C", "/content/Kabyle", "fetch", "origin", "main"],
    check=True,
)
subprocess.run(
    ["git", "-C", "/content/Kabyle", "checkout",
     "FETCH_HEAD", "--", "training/finetune.py"],
    check=True,
)
print("Correction installée.")

## Comparer avant et après
Le score chrF mesure la proximité des traductions de test. Ce n’est pas une mesure de qualité conversationnelle. Fais relire les sorties par un locuteur kabyle, teste des questions inconnues et compare au modèle de base avant de considérer l’adaptation comme une amélioration.


In [ ]:
evaluation = json.loads((output/'evaluation.json').read_text())
print(json.dumps(evaluation, ensure_ascii=False, indent=2))


In [ ]:
import shutil
adapter = output/'adapter'
training = json.loads((adapter/'training_manifest.json').read_text())
assert training['status'] == 'completed'
archive_path = shutil.make_archive('/content/kabyle-chat-adapter', 'zip', root_dir=output, base_dir='adapter')
files.download(archive_path)


## Utiliser sur le site
Sur ton PC, décompresse `kabyle-chat-adapter.zip` dans `artifacts/chat-model/` : le fichier doit se trouver à `artifacts/chat-model/adapter/training_manifest.json`.

Une seule fois : `bash scripts/install_chat_model.sh`.

Puis, dans un terminal séparé : `bash scripts/start_chat_model.sh`. Le premier démarrage télécharge le modèle de base et nécessite Internet. Le serveur reste sur `127.0.0.1:8010`, sans tunnel public. Lance le site normalement avec `bash scripts/start.sh`, puis ouvre **Parler kabyle**. Le CPU peut être lent.
